In [ ]:
# Block 1: Imports and Setup

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
INPUT_CSV_PATH = 'stockprice.csv'  # <-- Update this path if needed

# Identify the directory of the input file to create output folders there
BASE_DIR = os.path.dirname(os.path.abspath(INPUT_CSV_PATH)) if os.path.exists(INPUT_CSV_PATH) else '.'

# Create output directories
OUTPUT_FIGURES_DIR = os.path.join(BASE_DIR, 'output_figures')
OUTPUT_MODELS_DIR = os.path.join(BASE_DIR, 'trained_models')
OUTPUT_RESULTS_DIR = os.path.join(BASE_DIR, 'results')

os.makedirs(OUTPUT_FIGURES_DIR, exist_ok=True)
os.makedirs(OUTPUT_MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

print(f"Input CSV: {INPUT_CSV_PATH}")
print(f"Figures will be saved to: {OUTPUT_FIGURES_DIR}")
print(f"Models will be saved to: {OUTPUT_MODELS_DIR}")
print(f"Results will be saved to: {OUTPUT_RESULTS_DIR}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# --- Model Names for Consistency ---
MODEL_TYPES = ['Transformer', 'LSTM', 'RNN']

In [ ]:
# Block 2: Load and Inspect Data

# Load the dataset
try:
    df = pd.read_csv(INPUT_CSV_PATH)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: File {INPUT_CSV_PATH} not found.")
    raise

# Basic data cleaning and preparation
df.columns = df.columns.str.strip() # Remove potential leading/trailing spaces
df['trading date'] = pd.to_datetime(df['trading date'])
df = df.sort_values(['sector', 'trading date']).reset_index(drop=True)

# --- Dataset Overview ---
print("\n--- Dataset Info ---")
print(df.info())
print("\n--- First few rows ---")
print(df.head())
print("\n--- Unique Sectors ---")
sectors = df['sector'].unique()
print(sectors)
print("\n--- Checking for missing values ---")
print(df.isnull().sum())

In [ ]:
# Block 3: Define time2vec Layer (For Transformer)

class Time2Vec(layers.Layer):
    def __init__(self, kernel_size=1, **kwargs):
        super(Time2Vec, self).__init__(**kwargs)
        self.k = kernel_size

    def build(self, input_shape):
        self.w_linear = self.add_weight(shape=(input_shape[-1], 1), initializer="uniform", trainable=True, name='w_linear')
        self.b_linear = self.add_weight(shape=(1,), initializer="uniform", trainable=True, name='b_linear')
        self.w_periodic = self.add_weight(shape=(input_shape[-1], self.k), initializer="uniform", trainable=True, name='w_periodic')
        self.b_periodic = self.add_weight(shape=(self.k,), initializer="uniform", trainable=True, name='b_periodic')
        super(Time2Vec, self).build(input_shape)

    def call(self, inputs):
        linear = tf.matmul(inputs, self.w_linear) + self.b_linear
        periodic = tf.math.sin(tf.matmul(inputs, self.w_periodic) + self.b_periodic)
        return tf.concat([linear, periodic], axis=-1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], self.k + 1)

In [ ]:
# Block 4: Data Preprocessing Function

def preprocess_sector_data(group):
    """Preprocesses data for a single sector group."""
    group = group.copy()
    cols = ['open', 'high', 'low', 'close', 'volume']
    
    # 1. Apply 10-day moving average smoothing
    smoothed = group[cols].rolling(window=10, min_periods=1).mean()
    
    # 2. Compute returns (differencing) on smoothed data
    for col in cols:
        group[f'return_{col}'] = smoothed[col].diff()
    
    # 3. Drop rows with NaN (from diff and rolling)
    group = group.dropna().reset_index(drop=True)
    return group

def create_sequences(data, seq_length=8):
    """Creates input sequences and corresponding labels."""
    X, y, last_closes = [], [], []
    feature_cols = [f'return_{f}' for f in ['open', 'high', 'low', 'close', 'volume']]
    
    for i in range(seq_length, len(data)):
        seq = data[feature_cols].iloc[i-seq_length:i].values
        label = data['return_close'].iloc[i]
        last_close = data['close'].iloc[i-1]
        
        X.append(seq)
        y.append(label)
        last_closes.append(last_close)
        
    return np.array(X), np.array(y), np.array(last_closes)

# Dictionary to hold scalers for each sector
scalers = {}
processed_data_dict = {}

# Process data for each sector
for sector in sectors:
    print(f"Preprocessing data for sector: {sector}")
    sector_data = df[df['sector'] == sector].copy()
    processed_data = preprocess_sector_data(sector_data)
    
    # Normalize features
    feature_cols = [f'return_{f}' for f in ['open', 'high', 'low', 'close', 'volume']]
    scaler = MinMaxScaler()
    processed_data[feature_cols] = scaler.fit_transform(processed_data[feature_cols])
    scalers[sector] = scaler
    
    # Create sequences
    X, y, last_closes = create_sequences(processed_data, seq_length=8)
    processed_data_dict[sector] = {
        'X': X, 'y': y, 'last_closes': last_closes, 'raw_data': processed_data
    }
    print(f"  -> Sequences created: {X.shape}, Labels: {y.shape}")

print("Preprocessing complete for all sectors.")

In [ ]:
# Block 5: Define Model Architectures

# --- Transformer Model (from previous code) ---
def build_transformer_model(seq_length=8, n_features=5, d_model=32, n_heads=2, d_ff=64, dropout=0.1):
    inputs = layers.Input(shape=(seq_length, n_features), name='sequence_input')
    
    time_embeddings = Time2Vec(kernel_size=1)(inputs)
    linear_proj = layers.Dense(d_model, name='feature_projection')(inputs)
    time_proj = layers.Dense(d_model, name='time_projection')(time_embeddings)
    x = layers.Add(name='feature_time_fusion')([linear_proj, time_proj])
    
    positions = tf.range(start=0, limit=seq_length, delta=1)
    pos_encoding_layer = layers.Embedding(input_dim=seq_length, output_dim=d_model, name='positional_encoding')
    pos_encodings = pos_encoding_layer(positions)
    x = layers.Add()([x, pos_encodings])
    
    attn_output = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model, name='multi_head_attention')(x, x)
    attn_output = layers.Dropout(dropout)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6, name='ln_after_attention')(attn_output + x)
    
    ffn_output = layers.Dense(d_ff, activation='relu', name='ffn_dense1')(out1)
    ffn_output = layers.Dense(d_model, name='ffn_dense2')(ffn_output)
    ffn_output = layers.Dropout(dropout)(ffn_output)
    out2 = layers.LayerNormalization(epsilon=1e-6, name='ln_after_ffn')(ffn_output + out1)
    
    pooled = layers.GlobalAveragePooling1D(name='global_avg_pool')(out2)
    dense1 = layers.Dense(32, activation='relu', name='final_dense1')(pooled)
    dropout_final = layers.Dropout(0.1, name='final_dropout')(dense1)
    outputs = layers.Dense(1, name='output_layer')(dropout_final)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Transformer_Stock_Predictor")
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# --- LSTM Model ---
def build_lstm_model(seq_length=8, n_features=5, lstm_units=50):
    model = models.Sequential(name="LSTM_Stock_Predictor")
    model.add(layers.LSTM(units=lstm_units, input_shape=(seq_length, n_features), return_sequences=False))
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.1))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# --- RNN Model ---
def build_rnn_model(seq_length=8, n_features=5, rnn_units=50):
    model = models.Sequential(name="RNN_Stock_Predictor")
    model.add(layers.SimpleRNN(units=rnn_units, input_shape=(seq_length, n_features), return_sequences=False))
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.1))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Test model creation
# test_model = build_transformer_model()
# test_model.summary()
# test_model = build_lstm_model()
# test_model.summary()
# test_model = build_rnn_model()
# test_model.summary()

In [ ]:
# Block 6: Train Models, Evaluate, and Store Results

# Dictionary to store results for all models and sectors
all_results = {model_type: {} for model_type in MODEL_TYPES}
model_histories = {model_type: {} for model_type in MODEL_TYPES}

# Hyperparameters (matching paper as closely as possible)
EPOCHS = 50
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5 # Add early stopping for robustness

# --- Training Loop for All Models and Sectors ---
for sector in sectors:
    print(f"\n{'='*15} Processing Sector: {sector} {'='*15}")
    
    # 1. Get processed data
    data_dict = processed_data_dict[sector]
    X, y, last_closes = data_dict['X'], data_dict['y'], data_dict['last_closes']
    scaler = scalers[sector]
    
    if len(X) == 0:
        print(f"Skipping {sector} due to insufficient data after preprocessing.")
        continue

    # 2. Split data (80:10:10)
    n_total = len(X)
    n_train = int(0.8 * n_total)
    n_val = int(0.1 * n_total)

    X_train, y_train = X[:n_train], y[:n_train]
    X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
    X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]
    last_closes_test = last_closes[n_train+n_val:]
    
    print(f"  Data Split -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

    # --- Loop through each model type ---
    for model_type in MODEL_TYPES:
        print(f"\n  --- Training {model_type} Model for {sector} ---")
        
        # 3. Build Model
        if model_type == 'Transformer':
            model = build_transformer_model()
        elif model_type == 'LSTM':
            model = build_lstm_model()
        elif model_type == 'RNN':
            model = build_rnn_model()
        
        # Early stopping callback
        early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

        # 4. Train Model
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop], # Use early stopping
            verbose=0 # Reduce output
        )
        model_histories[model_type][sector] = history
        
        # 5. Evaluate on Test Set
        y_pred_norm = model.predict(X_test, verbose=0).flatten()
        
        # 6. Inverse Transform Predictions and True Labels
        # Inverse transform the target (close return) using the scaler
        y_test_actual_returns = scaler.inverse_transform(
            np.hstack([np.zeros((len(y_test), 4)), y_test.reshape(-1, 1)])
        )[:, 4]

        y_pred_actual_returns = scaler.inverse_transform(
            np.hstack([np.zeros((len(y_pred_norm), 4)), y_pred_norm.reshape(-1, 1)])
        )[:, 4]

        # 7. Calculate Final Metrics on Actual Returns
        rmse_actual = np.sqrt(mean_squared_error(y_test_actual_returns, y_pred_actual_returns))
        mae_actual = mean_absolute_error(y_test_actual_returns, y_pred_actual_returns)
        
        # 8. Reconstruct Actual Closing Prices
        predicted_closing_prices = last_closes_test + y_pred_actual_returns
        actual_closing_prices = last_closes_test + y_test_actual_returns
        
        # 9. Store results
        all_results[model_type][sector] = {
            'RMSE_Actual_Return': rmse_actual, 'MAE_Actual_Return': mae_actual,
            'y_test_actual_returns': y_test_actual_returns,
            'y_pred_actual_returns': y_pred_actual_returns,
            'actual_closing_prices': actual_closing_prices,
            'predicted_closing_prices': predicted_closing_prices,
            'test_dates_indices': np.arange(n_train+n_val, n_total)
        }
        
        # 10. Save Model
        model_path = os.path.join(OUTPUT_MODELS_DIR, f"model_{model_type}_{sector.replace(' ', '_')}.keras")
        model.save(model_path)
        print(f"    -> {model_type} model saved to: {model_path}")
        print(f"    -> Test RMSE (Actual Returns): {rmse_actual:.6f}")
        print(f"    -> Test MAE (Actual Returns): {mae_actual:.6f}")

print("\nTraining and evaluation completed for all models and sectors.")

In [ ]:
# Block 7: Generate Comparative Results Table

# Create a summary DataFrame
results_summary_data = []
for model_type in MODEL_TYPES:
    for sector, metrics in all_results[model_type].items():
        results_summary_data.append({
            'Model': model_type,
            'Sector': sector,
            'Daily_RMSE': metrics['RMSE_Actual_Return'],
            'Daily_MAE': metrics['MAE_Actual_Return']
        })

results_df = pd.DataFrame(results_summary_data)
# Sort for consistent order
results_df = results_df.sort_values(['Sector', 'Model']).reset_index(drop=True)

# Save to CSV
results_csv_path = os.path.join(OUTPUT_RESULTS_DIR, "comparative_model_performance_summary.csv")
results_df.to_csv(results_csv_path, index=False)
print(f"Comparative results summary saved to: {results_csv_path}")

# Display a pivot table for easy comparison
pivot_rmse = results_df.pivot(index='Sector', columns='Model', values='Daily_RMSE')
pivot_mae = results_df.pivot(index='Sector', columns='Model', values='Daily_MAE')

print("\n--- RMSE Comparison (Lower is Better) ---")
print(pivot_rmse)
print("\n--- MAE Comparison (Lower is Better) ---")
print(pivot_mae)

# --- Research-Driven Analysis ---
print("\n--- Research-Driven Analysis ---")
for sector in sectors:
    if sector in all_results['Transformer']: # Check if model was trained
        print(f"\nSector: {sector}")
        sector_results = {model: all_results[model][sector] for model in MODEL_TYPES if sector in all_results[model]}
        
        best_rmse_model = min(sector_results, key=lambda m: sector_results[m]['RMSE_Actual_Return'])
        best_mae_model = min(sector_results, key=lambda m: sector_results[m]['MAE_Actual_Return'])
        
        print(f"  Best RMSE Model: {best_rmse_model} ({sector_results[best_rmse_model]['RMSE_Actual_Return']:.6f})")
        print(f"  Best MAE Model: {best_mae_model} ({sector_results[best_mae_model]['MAE_Actual_Return']:.6f})")
        
        # Example: Compare Transformer vs LSTM
        if 'Transformer' in sector_results and 'LSTM' in sector_results:
            trans_rmse = sector_results['Transformer']['RMSE_Actual_Return']
            lstm_rmse = sector_results['LSTM']['RMSE_Actual_Return']
            diff_rmse = trans_rmse - lstm_rmse
            if diff_rmse < 0:
                print(f"  -> Transformer outperformed LSTM by {-diff_rmse:.6f} RMSE.")
            else:
                print(f"  -> LSTM outperformed Transformer by {diff_rmse:.6f} RMSE.")

In [ ]:
# Block 8: Generate Graphs (Following Paper's Figures)

def plot_training_history(history, model_type, sector_name, save_path):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.history['loss'], label='Training Loss')
    ax.plot(history.history['val_loss'], label='Validation Loss')
    ax.set_title(f'{model_type} Model Loss for {sector_name}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

def plot_price_prediction(actual_prices, predicted_prices, model_type, sector_name, save_path, num_points=100):
    if len(actual_prices) > num_points:
        actual_prices = actual_prices[-num_points:]
        predicted_prices = predicted_prices[-num_points:]
        
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(actual_prices, label='Actual Closing Price', marker='o', markersize=3)
    ax.plot(predicted_prices, label=f'Predicted Closing Price ({model_type})', marker='x', markersize=3)
    ax.set_title(f'{model_type} Model: Closing Price Prediction vs Actual - {sector_name} (Last {len(actual_prices)} Days)')
    ax.set_xlabel('Time (Test Set Index)')
    ax.set_ylabel('Closing Price')
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

def plot_comparative_price_prediction(sector_results, sector_name, save_path, num_points=100):
    """Plot predictions of all models for a single sector."""
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Get the longest actual price series to plot as the base
    base_model = list(sector_results.keys())[0]
    actual_prices_full = sector_results[base_model]['actual_closing_prices']
    
    if len(actual_prices_full) > num_points:
        actual_prices = actual_prices_full[-num_points:]
        start_idx = len(actual_prices_full) - num_points
    else:
        actual_prices = actual_prices_full
        start_idx = 0
    
    ax.plot(actual_prices, label='Actual Closing Price', marker='o', linewidth=2, markersize=4)
    
    for model_type, metrics in sector_results.items():
        pred_prices_full = metrics['predicted_closing_prices']
        if len(pred_prices_full) >= len(actual_prices_full):
            pred_prices = pred_prices_full[start_idx:]
        else:
            # Handle potential mismatch in sequence lengths (shouldn't happen with consistent preprocessing)
            pred_prices = pred_prices_full[-len(actual_prices):] if len(pred_prices_full) >= len(actual_prices) else pred_prices_full
        
        ax.plot(pred_prices, label=f'Predicted ({model_type})', marker='x', linestyle='--', markersize=4)
        
    ax.set_title(f'Model Comparison: Closing Price Prediction for {sector_name} (Last {len(actual_prices)} Days)')
    ax.set_xlabel('Time (Test Set Index)')
    ax.set_ylabel('Closing Price')
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

# --- Generate Graphs for Each Sector and Model ---
for sector in sectors:
    sector_graph_dir = os.path.join(OUTPUT_FIGURES_DIR, sector.replace(' ', '_'))
    os.makedirs(sector_graph_dir, exist_ok=True)
    
    sector_results = {model: all_results[model][sector] for model in MODEL_TYPES if sector in all_results[model]}
    
    if not sector_results:
        continue

    # 1. Individual Model Training History Plots
    for model_type in MODEL_TYPES:
        if model_type in model_histories and sector in model_histories[model_type]:
            history = model_histories[model_type][sector]
            history_plot_path = os.path.join(sector_graph_dir, f"history_{model_type}_{sector.replace(' ', '_')}.png")
            plot_training_history(history, model_type, sector, history_plot_path)
            print(f"Saved training history plot for {model_type} - {sector}")

    # 2. Individual Model Prediction vs Actual Price Plots
    for model_type, metrics in sector_results.items():
        price_plot_path = os.path.join(sector_graph_dir, f"prediction_{model_type}_{sector.replace(' ', '_')}.png")
        plot_price_prediction(
            metrics['actual_closing_prices'], 
            metrics['predicted_closing_prices'], 
            model_type, 
            sector, 
            price_plot_path
        )
        print(f"Saved prediction plot for {model_type} - {sector}")

    # 3. Comparative Prediction Plot (All models for this sector)
    comparative_plot_path = os.path.join(sector_graph_dir, f"comparison_all_models_{sector.replace(' ', '_')}.png")
    plot_comparative_price_prediction(sector_results, sector, comparative_plot_path)
    print(f"Saved comparative prediction plot for {sector}")

print("All graphs generated and saved.")

In [ ]:
# Block 9: (Optional) Weekly Model Extension
# This block is a placeholder. Implementation would follow the same structure:
# 1. Resample daily data to weekly OHLCV.
# 2. Apply the same preprocessing (MA smoothing, differencing, normalization).
# 3. Create sequences (e.g., last 8 weeks to predict next week).
# 4. Train Transformer, LSTM, RNN models on weekly data.
# 5. Evaluate and store results in a similar `all_results_weekly` dictionary.
# 6. Generate comparative tables and graphs for weekly models.
# 7. Integrate weekly results into the final analysis if desired.
#
# Due to length and focus on daily models as per the paper's primary experiment,
# this is not implemented here but follows the same blueprint.